# 03 — From Embeddings to Predictions: the Linear Probe

**By the end of this notebook** you'll have extracted `[USR]` embeddings from the PRAGMA model, trained a linear probe on a synthetic binary classification task, visualised the embedding space with PCA, and measured how much the probe improves over a random-weight baseline.

## What this notebook teaches

- What a linear probe is and why it is the standard way to evaluate representation quality (§3.1.1)
- How to extract the `[USR]` embedding from `output["zh"][:, 0, :]` after a PRAGMA forward pass
- How `EmbeddingProbe` fits a logistic regression on frozen embeddings
- How to read a 2D PCA scatter plot of the embedding space coloured by class label
- Why a trained model produces more separable embeddings than a random-weight baseline

## Prerequisites

Run notebooks 01 and 02 first. This notebook reuses the same `VocabularySpec` and batch construction from notebook 02, and trains a short loop before probing.

**How to use:** run every cell in order with Shift+Enter.

In [ ]:
# ── Setup: repo root on sys.path ─────────────────────────────────────────────
import sys, pathlib

# Locate repo root by searching candidate paths for pyproject.toml.
# Works whether Jupyter CWD is:
#   - the repo root itself           (local dev, top-level launch)
#   - notebooks/                     (local dev, launched from subdir)
#   - /opt/app-root/src              (OpenShift AI Workbench — CWD is parent of repo)
#   - /opt/app-root/src/pragma-encoder  (workbench with repo as CWD)
_here = pathlib.Path().resolve()
repo_root = next(
    (p for p in [_here, _here.parent,
                 _here / "pragma-encoder", _here.parent / "pragma-encoder"]
     if (p / "pyproject.toml").exists()),
    _here,
)
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch

# ── PRAGMA model ──────────────────────────────────────────────────────────────
from pragma_encoder.model           import PRAGMA, PRAGMAConfig
from pragma_encoder.model.assembler import EmbeddingAssembler
from pragma_encoder.masking         import MaskingStrategy
from pragma_encoder.tokenizer.vocabulary import VocabularySpec

# ── PRAGMA linear probe (src/pragma_encoder/adaptation/probe.py) ──────────────
from pragma_encoder.adaptation.probe import EmbeddingProbe   # §3.1.1

matplotlib.rcParams['figure.dpi'] = 110
torch.manual_seed(42)
np.random.seed(42)
print(f"Setup complete. repo_root={repo_root}")

## What is a linear probe?

A **linear probe** tests how much useful information a frozen representation already contains. The procedure is:

1. Freeze the encoder — no gradient updates
2. Run a labelled dataset through the encoder to get one embedding vector per sample
3. Fit a logistic regression on those embeddings
4. Measure AUC-ROC on a held-out test set

If the probe achieves high AUC with a *linear* classifier, it means the encoder has already separated the classes in embedding space — the representation is good. If the probe barely beats random, the encoder hasn't learned the right structure yet.

PRAGMA uses the `[USR]` token output from the History Encoder as the customer representation (§3.1.1). The `[USR]` token sits at position 0 of `output["zh"]`, so the embedding is `output["zh"][:, 0, :]`.

Source: `src/pragma_encoder/adaptation/probe.py::EmbeddingProbe`

## Synthetic dataset with class labels

We need customers with labels. We'll create 60 customers split into two synthetic classes. Class 1 customers have systematically higher transaction values — a deliberate pattern the model can (in principle) pick up after training.

In [ ]:
# ── Vocabulary (same spec as notebook 02) ────────────────────────────────────
config     = PRAGMAConfig.pragma_s()
vocab_spec = VocabularySpec(
    special_tokens={"PAD": 0, "MASK": 1, "EVT": 2, "SEP": 3},
    key_start=4,
    key_size=config.key_vocab_size,
    value_start=4 + config.key_vocab_size,
    value_size=config.value_vocab_size,
    total_embedding_vocab_size=4 + config.key_vocab_size + config.value_vocab_size,
    field_key_ids={},
    field_value_ranges={},
)

# ── Batch dimensions ──────────────────────────────────────────────────────────
N_CUSTOMERS = 60    # 48 train, 12 test
NE, NI, NA  = 10, 8, 6

# ── Generate two synthetic classes ────────────────────────────────────────────
# Class 0: value IDs drawn from the lower half of the value range
# Class 1: value IDs drawn from the upper half of the value range
# This gives the model a learnable signal if it trains long enough.
rng_g = torch.Generator()
rng_g.manual_seed(7)

labels   = torch.tensor([i % 2 for i in range(N_CUSTOMERS)], dtype=torch.long)  # 0/1 alternating
mid_val  = vocab_spec.value_start + vocab_spec.value_size // 2

def _make_val_ids_for_customer(label: int) -> torch.Tensor:
    """Class 0 → lower value IDs; class 1 → upper value IDs."""
    if label == 0:
        return torch.randint(vocab_spec.value_start, mid_val, (NE, NI), generator=rng_g)
    else:
        return torch.randint(mid_val, vocab_spec.value_start + vocab_spec.value_size, (NE, NI), generator=rng_g)

xe_val_ids = torch.stack([_make_val_ids_for_customer(int(l)) for l in labels])
xe_key_ids = torch.randint(vocab_spec.key_start, vocab_spec.key_start + vocab_spec.key_size,
                            (N_CUSTOMERS, NE, NI), generator=rng_g)
xa_key_ids = torch.randint(vocab_spec.key_start, vocab_spec.key_start + vocab_spec.key_size,
                            (N_CUSTOMERS, NA), generator=rng_g)
xa_val_ids = torch.randint(vocab_spec.value_start, vocab_spec.value_start + vocab_spec.value_size,
                            (N_CUSTOMERS, NA), generator=rng_g)
ta         = torch.rand(N_CUSTOMERS, NA, generator=rng_g) * 5.0
te         = torch.rand(N_CUSTOMERS, NE, generator=rng_g) * 80.0
calendar   = torch.rand(N_CUSTOMERS, NE, 3, generator=rng_g)

print(f"{N_CUSTOMERS} customers: {(labels==0).sum().item()} class-0, {(labels==1).sum().item()} class-1")
print(f"xe_val_ids shape: {tuple(xe_val_ids.shape)}")

## Utility: extract embeddings

The function below runs a batch of customers through a frozen PRAGMA model and extracts the `[USR]` embedding from `output["zh"][:, 0, :]`.

In [ ]:
def extract_usr_embeddings(
    mdl: PRAGMA, asm: EmbeddingAssembler, batch_size: int = 12
) -> torch.Tensor:
    """Extract [USR] embeddings for all N_CUSTOMERS in mini-batches.

    [USR] is at position 0 of output["zh"]  (§3.1.1, History Encoder output).
    Returns shape: (N_CUSTOMERS, d_model).
    """
    mdl.eval()
    all_emb = []
    with torch.no_grad():
        for start in range(0, N_CUSTOMERS, batch_size):
            end = min(start + batch_size, N_CUSTOMERS)
            assembled = asm.forward(
                xa_key_ids=xa_key_ids[start:end],
                xa_val_ids=xa_val_ids[start:end],
                ta=ta[start:end],
                xe_key_ids=xe_key_ids[start:end],
                xe_val_ids=xe_val_ids[start:end],
                te=te[start:end],
                calendar=calendar[start:end],
                targets=xe_val_ids[start:end],
                mlm_mask=torch.zeros(end - start, NE, NI, dtype=torch.bool),
            )
            out = mdl.forward(
                xa=assembled.xa, ta=assembled.ta,
                xe=assembled.xe, xt=assembled.xt,
                te=assembled.te, mask=assembled.mlm_mask,
            )
            # [USR] token is at position 0 of the History Encoder output zh
            usr_emb = out["zh"][:, 0, :]   # (batch, d_model)
            all_emb.append(usr_emb.cpu())
    return torch.cat(all_emb, dim=0)   # (N_CUSTOMERS, d_model)

print(f"EmbeddingProbe will receive (N_CUSTOMERS, d_model) = ({N_CUSTOMERS}, {config.d_model})")

## Baseline — random-weight model

Before training anything, let's measure what a randomly-initialised PRAGMA model achieves as a probe baseline. If our trained model is better, it's because pretraining actually learned something.

In [ ]:
# ── Probe train/test split ────────────────────────────────────────────────────
# 80% train, 20% test — stratified by alternating labels
train_idx = list(range(0, N_CUSTOMERS, 5)) + list(range(1, N_CUSTOMERS, 5)) + \
            list(range(2, N_CUSTOMERS, 5)) + list(range(3, N_CUSTOMERS, 5))
test_idx  = list(range(4, N_CUSTOMERS, 5))
train_idx.sort()

train_labels = labels[train_idx]
test_labels  = labels[test_idx]

# ── Random baseline ───────────────────────────────────────────────────────────
torch.manual_seed(0)
model_rand  = PRAGMA(config)                           # random weights
asm_rand    = EmbeddingAssembler(vocab_spec, config)   # random embedding table

emb_rand = extract_usr_embeddings(model_rand, asm_rand)

probe_rand = EmbeddingProbe(config)
probe_rand.fit(emb_rand[train_idx], train_labels, task="classification")
auc_rand   = probe_rand.score(emb_rand[test_idx], test_labels)

print(f"Baseline (random weights) probe AUC-ROC: {auc_rand:.3f}")
print("  (0.5 = random chance; 1.0 = perfect separation)")

## Train PRAGMA-S for 50 steps, then probe

In [ ]:
# ── Training setup ────────────────────────────────────────────────────────────
N_STEPS    = 50
BATCH_SIZE = 12

torch.manual_seed(42)
model     = PRAGMA(config)
assembler = EmbeddingAssembler(vocab_spec, config)
masker    = MaskingStrategy(config)
optimizer = torch.optim.Adam(
    list(model.parameters()) + list(assembler.parameters()), lr=1e-4
)

losses = []

def _mini_batch(idx_list):
    """Select a random mini-batch of BATCH_SIZE customers from idx_list."""
    chosen = torch.randperm(len(idx_list))[:BATCH_SIZE].tolist()
    idxs   = [idx_list[i] for i in chosen]
    return {
        "xa_key_ids": xa_key_ids[idxs],
        "xa_val_ids": xa_val_ids[idxs],
        "ta":         ta[idxs],
        "xe_key_ids": xe_key_ids[idxs],
        "xe_val_ids": xe_val_ids[idxs],
        "te":         te[idxs],
        "calendar":   calendar[idxs],
    }

print(f"Pretraining PRAGMA-S for {N_STEPS} steps...")
for step in range(N_STEPS):
    mb = _mini_batch(train_idx)
    optimizer.zero_grad()
    masked_vals, _, mlm_mask = masker.forward(mb["xe_val_ids"], mb["xe_key_ids"])
    if not mlm_mask.any():
        mlm_mask[0, 0, 0]   = True
        masked_vals[0, 0, 0] = vocab_spec.special_tokens["MASK"]
    assembled = assembler.forward(
        xa_key_ids=mb["xa_key_ids"], xa_val_ids=mb["xa_val_ids"], ta=mb["ta"],
        xe_key_ids=mb["xe_key_ids"], xe_val_ids=masked_vals, te=mb["te"],
        calendar=mb["calendar"], targets=mb["xe_val_ids"], mlm_mask=mlm_mask,
    )
    out  = model.forward(xa=assembled.xa, ta=assembled.ta, xe=assembled.xe,
                         xt=assembled.xt, te=assembled.te, mask=assembled.mlm_mask)
    loss = model.mlm_head.compute_loss(out["logits"], assembled.targets[assembled.mlm_mask])
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step == 0 or (step + 1) % 10 == 0:
        print(f"  step {step+1:3d}  loss={loss.item():.4f}")

print("Pretraining complete.")

In [ ]:
# ── Extract embeddings from trained model ─────────────────────────────────────
emb_trained = extract_usr_embeddings(model, assembler)

# ── Fit and score the probe ───────────────────────────────────────────────────
probe_trained = EmbeddingProbe(config)
probe_trained.fit(emb_trained[train_idx], train_labels, task="classification")
auc_trained   = probe_trained.score(emb_trained[test_idx], test_labels)

print(f"Trained model probe AUC-ROC  : {auc_trained:.3f}")
print(f"Random baseline probe AUC-ROC: {auc_rand:.3f}")
print(f"Delta                         : {auc_trained - auc_rand:+.3f}")

## Visualisation 1 — PCA of embedding space

A 2D PCA projects the high-dimensional `[USR]` embeddings onto the two axes of greatest variance. Coloured by class label, this reveals whether the model has separated the two classes in embedding space.

In [ ]:
def pca_2d(emb: torch.Tensor) -> np.ndarray:
    """Manual 2-component PCA — no sklearn dependency."""
    X     = emb.numpy().astype(np.float64)
    X     = X - X.mean(axis=0, keepdims=True)
    cov   = (X.T @ X) / (X.shape[0] - 1)
    eigvals, eigvecs = np.linalg.eigh(cov)
    top2  = eigvecs[:, np.argsort(eigvals)[::-1][:2]]
    return X @ top2   # (N, 2)

proj_rand    = pca_2d(emb_rand)
proj_trained = pca_2d(emb_trained)
label_arr    = labels.numpy()
colors       = ["#4C72B0" if l == 0 else "#DD8452" for l in label_arr]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, proj, title in [
    (ax1, proj_rand,    f"Random weights"),
    (ax2, proj_trained, f"After {N_STEPS}-step pretraining"),
]:
    ax.scatter(proj[:, 0], proj[:, 1], c=colors, s=60, alpha=0.8, edgecolors="white", linewidths=0.5)
    # Legend patches
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color="#4C72B0", label="class 0"),
                        Patch(color="#DD8452", label="class 1")],
              fontsize=9, loc="best")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("PC 1", fontsize=10)
    ax.set_ylabel("PC 2", fontsize=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle("[USR] embedding PCA — before vs after pretraining (§3.1.1)", fontsize=12)
plt.tight_layout()
plt.show()

**What you're looking at:** each dot is one customer's `[USR]` embedding projected onto its top two principal components. Blue = class 0 (lower-value transactions), orange = class 1 (higher-value transactions). After 50 pretraining steps the two classes should be more separated than in the random-weight baseline — the model has begun to learn that certain value token ranges co-occur in certain customers.

## Visualisation 2 — Probe AUC: random baseline vs trained

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

labels_bar  = ["Random\nbaseline", f"After {N_STEPS}\nsteps"]
aucs        = [auc_rand, auc_trained]
bar_colors  = ["#C44E52", "#55A868"]

bars = ax.bar(labels_bar, aucs, color=bar_colors, width=0.45,
              edgecolor="white", linewidth=1.5)
ax.axhline(0.5, color="#888", linewidth=1.2, linestyle="--", label="random chance (0.5)")

for bar, auc in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{auc:.3f}", ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.set_ylabel("AUC-ROC (linear probe, §3.1.1)", fontsize=11)
ax.set_title("Linear probe: does pretraining improve embedding quality?", fontsize=11)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

print(f"\nConclusion: {'trained > baseline ✓' if auc_trained > auc_rand else 'no improvement yet — more steps needed'}")

**What you're looking at:** both bars show AUC-ROC for a logistic regression trained on frozen embeddings. The dashed line at 0.5 is random chance. If the trained model's bar is higher, pretraining has given the embeddings structure the probe can exploit. On 50 steps of synthetic data the gap is small — but it demonstrates the mechanism. On thousands of real TabFormer customers trained to convergence the gap is large.

## What just happened — section recap

- **EmbeddingProbe** freezes the encoder, extracts `[USR]` embeddings (`output["zh"][:, 0, :]`), fits `LogisticRegression`, and measures AUC-ROC
- A random-weight model produces embeddings that carry no class information — the probe scores near 0.5
- After even a short pretraining loop the embeddings begin to separate the classes
- PCA makes this visible: the cluster structure sharpens after pretraining

## Closing — what you now know

**Linear probe:** freeze encoder → extract `[USR]` → fit logistic regression → measure AUC. This is the standard benchmark for representation quality.

**Embedding extraction:** `output["zh"][:, 0, :]` — position 0 is always the `[USR]` token output from the History Encoder.

**`EmbeddingProbe` API:** `probe.fit(emb_train, labels, task="classification")` → `probe.score(emb_test, labels)`.

**Next:** notebook 04 applies LoRA fine-tuning (§3.1.2) to the same model — updating only ~2–4% of parameters — and compares its accuracy to this frozen probe.

**Going deeper:** `src/pragma_encoder/adaptation/probe.py` — the full `EmbeddingProbe` implementation. `docs/paper-to-code.md §3.1.1` — paper-to-code mapping for the probe.